# 10차시 실습 — 기억하고 찾아오는 에이전트 (메모리 + 임베딩 검색)

> **day1 연속 실습.** day1 추천 앱 → 7차시 **ReAct 에이전트** → 9차시 **도구·MCP** 에 이어,
> 오늘은 **같은 맛집 추천 에이전트**에 **기억(memory)** 과 **내 데이터 검색(RAG)** 을 더합니다.

1. **대화 메모리** — 이어지는 대화를 기억하고, 길어지면 **요약**으로 압축
2. **임베딩 검색** — 우리 서비스 데이터(맛집 메모)를 의미로 검색(코사인 유사도, top-k)
3. **근거 있는 답(RAG)** — 검색해 온 메모**만** 근거로 추천하기

## 🧪 사용법
- 워크샵 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**해 보는 동반 자료입니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다.
- 위에서부터 **순서대로** 실행하세요 (`Shift+Enter`).

In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
import numpy as np
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오

7·9차시에서 **맛집 추천 에이전트**를 만들었습니다(ReAct 루프 + 도구).
하지만 그 에이전트는 — **지난 대화도, 우리가 쌓아둔 맛집 메모도** 기억·참조하지 못합니다.

오늘 같은 에이전트에 **① 대화 기억**과 **② 내 데이터(맛집 메모) 검색**을 붙입니다.

> 구조는 동일하니 **당신 팀 MVP의 데이터**로 바꾸면 그대로 적용됩니다(8단계).

## 1단계 — 대화 메모리 (단기 기억)

LLM은 매 호출이 **백지**입니다. 그래서 '기억하는 것처럼' 보이려면 **지금까지의 대화 전체를 매번 다시 넣어줘야** 합니다.
아래 `chat_history`(메시지 목록)가 곧 에이전트의 **단기 메모리**입니다.

⌨️ **터미널 Codex:** `codex "대화를 누적 리스트로 기억하는 한국어 추천 챗봇 루프를 만들어줘"`

In [2]:
# chat_history 한 줄에 대화가 쌓입니다 = 단기 메모리
chat_history = [
    {"role": "system", "content": "너는 맛집 추천 도우미다. 대화 맥락(이름·취향·예산)을 기억해서 답하라."}
]

def say(user_msg, verbose=True):
    chat_history.append({"role": "user", "content": user_msg})
    resp = client.chat.completions.create(model=MODEL, messages=chat_history)
    reply = resp.choices[0].message.content
    chat_history.append({"role": "assistant", "content": reply})  # 답도 기억에 추가
    if verbose:
        print(f"🙂 나: {user_msg}")
        print(f"🤖 봇: {reply}\n")
    return reply

say("안녕! 내 이름은 지노이고, 매운 음식을 좋아해. 예산은 2만원이야.")
say("내 이름이 뭐였지? 그리고 내 취향·예산 기억해?")   # ← 앞 대화를 '기억'해서 답함

🙂 나: 안녕! 내 이름은 지노이고, 매운 음식을 좋아해. 예산은 2만원이야.
🤖 봇: 안녕하세요, 지노님! 매운 음식을 좋아하신다고 하니, 매운 떡볶이나 초중국식 매운탕 같은 메뉴가 좋을 것 같아요. 예산이 2만원인 점을 고려해서 추천해 드릴게요.

1. **떡볶이 전문점**: 매운 떡볶이를 파는 곳에서 추가로 오뎅이나 튀김을 시켜보세요. 종류에 따라 매운 정도를 조절할 수 있는 곳도 많습니다.

2. **매운 닭발**: 매운 닭발을 전문으로 하는 식당도 추천드립니다. 술안주로도 좋고, 밥과 함께 먹으면 맛있습니다.

3. **매운 국 또는 찌개**: 매운 해물탕이나 김치찌개 같은 것을 파는 식당도 좋습니다. 찌개와 밥을 함께 시켜서 드시면 든든하게 식사할 수 있을 거예요.

이 외에도 지역에 따라 다양한 매운 음식을 제공하는 곳이 있으니, 근처 검색해 보시면 좋을 것 같습니다. 맛있는 매운 음식 드세요!

🙂 나: 내 이름이 뭐였지? 그리고 내 취향·예산 기억해?
🤖 봇: 네, 지노님! 매운 음식을 좋아하시고 예산은 2만원이라고 기억하고 있습니다. 추가적으로 도움이 필요하시면 말씀해 주세요!



'네, 지노님! 매운 음식을 좋아하시고 예산은 2만원이라고 기억하고 있습니다. 추가적으로 도움이 필요하시면 말씀해 주세요!'

### 1단계+ — 요약으로 압축하기 (장기 기억의 첫걸음)

대화가 길어지면 컨텍스트 윈도우가 가득 차 **오래된 내용부터 사라집니다(FIFO)**.
그래서 긴 대화는 **요약(summary)** 으로 압축해, 핵심 사실(이름·취향·예산)만 오래 보존합니다.

In [3]:
def summarize(history):
    convo = "\n".join(f"{m['role']}: {m['content']}" for m in history if m["role"] != "system")
    resp = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "다음 대화를 한국어 2~3문장으로 요약하라. 사용자 핵심 사실(이름·취향·예산)은 반드시 보존하라."},
        {"role": "user", "content": convo},
    ])
    return resp.choices[0].message.content

print("📝 대화 요약:")
print(summarize(chat_history))

📝 대화 요약:
지노라는 이름을 가진 사용자님은 매운 음식을 좋아하며, 예산은 2만원입니다. 이 정보를 기반으로 추천을 할 수 있습니다.


## 2단계 — 임베딩 검색 (우리 데이터를 의미로 찾기)

이제 **지식**입니다. 모델이 모르는 '우리 서비스 데이터'(아래 맛집 메모)를 **임베딩**(의미를 담은 숫자 벡터)으로 바꾸고,
질문과의 **코사인 유사도**로 가장 관련 있는 메모를 찾습니다 — 단어가 안 겹쳐도 **의미가 가까우면** 찾아냅니다.

⌨️ **터미널 Codex:** `codex "맛집 메모를 text-embedding-3-large로 임베딩하고 코사인 유사도로 top-k 검색하는 함수를 만들어줘"`

In [9]:
# 우리 서비스의 데이터 = 맛집 메모 (7차시 RESTAURANTS의 '리뷰 노트' 버전)
notes = [
    "매운갈비찜집: 강남. 갈비찜이 칼칼하고 양 많음. 1인 1.8만원. 주차 가능, 회식 좋음.",
    "순한국밥집: 강남. 담백한 소고기국밥, 해장에 좋음. 9천원. 24시간 영업.",
    "불막창: 강남. 막창구이가 매콤, 단체석 있음. 1인 2.2만원. 예약 권장.",
    "치즈파스타: 홍대. 크림파스타가 부드럽고 분위기 좋아 데이트에 좋음. 1.5만원.",
    "라멘하루: 홍대. 돈코츠 라멘이 진하고 혼밥하기 편함. 1.1만원. 웨이팅 있음.",
]

def embed(texts):
    resp = client.embeddings.create(model="text-embedding-3-large", input=texts)
    return np.array([d.embedding for d in resp.data])

note_vecs = embed(notes)   # 메모들을 미리 임베딩 = 인제스션(한 번만)

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def search(query, k=6):
    qv = embed([query])[0]                          # 질문도 같은 모델로 임베딩
    scores = [cosine(qv, dv) for dv in note_vecs]   # 모든 메모와 유사도 계산
    order = np.argsort(scores)[::-1][:k]            # 점수 높은 순 top-k
    return [(notes[i], scores[i]) for i in order]

# 단어가 안 겹쳐도(혼밥≠혼자) 의미로 찾아옵니다
for note, score in search("혼자 가서 편하게 먹을 만한 곳 있어?", k=2):
    print(f"{score:.3f}  {note}")

0.391  라멘하루: 홍대. 돈코츠 라멘이 진하고 혼밥하기 편함. 1.1만원. 웨이팅 있음.
0.380  순한국밥집: 강남. 담백한 소고기국밥, 해장에 좋음. 9천원. 24시간 영업.


## 3단계 — 근거 있는 답 (RAG = 검색 + 생성)

검색해 온 메모**만**을 근거로 추천하면, 모델이 **지어내지 않고(환각↓)** 우리 데이터에 맞춰 답합니다.
이것이 **RAG**의 핵심: `질문 → 검색(R) → (질문+메모) → 생성(G) → 근거 있는 답`.

⌨️ **터미널 Codex:** `codex "검색된 맛집 메모만 근거로 추천하고 출처를 함께 보여주는 RAG 함수를 만들어줘"`

In [10]:
def answer(query, k=2):
    hits = search(query, k)
    context = "\n".join(f"- {n}" for n, _ in hits)
    resp = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "아래 <맛집 메모>의 내용만 근거로 한국어로 추천하라. 메모에 없는 내용은 '메모에 없습니다'라고 답하라."},
        {"role": "user", "content": f"<맛집 메모>\n{context}\n</맛집 메모>\n\n질문: {query}"},
    ])
    print("📚 근거로 찾은 메모:")
    for n, s in hits:
        print(f"  ({s:.3f}) {n}")
    print(f"\n🤖 추천: {resp.choices[0].message.content}")

answer("강남에서 해장하기 좋은 곳 추천해줘")   # '해장'이 들어간 메모를 찾아 근거로 답함

📚 근거로 찾은 메모:
  (0.500) 순한국밥집: 강남. 담백한 소고기국밥, 해장에 좋음. 9천원. 24시간 영업.
  (0.494) 불막창: 강남. 막창구이가 매콤, 단체석 있음. 1인 2.2만원. 예약 권장.

🤖 추천: 강남에서 해장하기 좋은 곳으로는 '순한국밥집'을 추천합니다. 이곳은 담백한 소고기국밥을 제공하며, 해장에 좋습니다. 가격은 9천원이고, 24시간 영업하니 언제든지 방문할 수 있습니다.


## 4단계 — 직접 바꿔보기 (iterate)

- **메모를 추가**해 보세요 → `note_vecs`를 다시 임베딩해야 검색에 반영됩니다.
- **질문을 바꿔** 보세요: `answer("데이트하기 좋은 곳은?")`, `answer("회사 강아지 이름은?")` (← 메모에 없으면 '없습니다')

⌨️ **터미널 Codex:** `codex "notes에 새 맛집 메모를 추가하고 다시 임베딩해서 검색되게 해줘"`

In [11]:
# 새 메모 추가 → 다시 임베딩(인제스션) → 검색에 반영
notes.append("비건테이블: 홍대. 채식 위주, 알레르기 대응 가능, 조용함. 1.4만원.")
note_vecs = embed(notes)

answer("고기 안 먹는 친구랑 갈 만한 곳?")   # '채식/비건'이란 단어가 질문에 없어도 의미로 찾아옴

📚 근거로 찾은 메모:
  (0.380) 비건테이블: 홍대. 채식 위주, 알레르기 대응 가능, 조용함. 1.4만원.
  (0.363) 순한국밥집: 강남. 담백한 소고기국밥, 해장에 좋음. 9천원. 24시간 영업.

🤖 추천: 비건테이블을 추천합니다. 홍대에 위치하고, 채식 위주이며 알레르기 대응이 가능하고 조용한 분위기입니다. 가격은 1.4만원입니다.


## 5단계 — 망각 전략 비교: 잘라내기 vs 요약

윈도우가 가득 찼다고 가정하고, **Trimming**(최근 몇 턴만)과 **Summarization**(요약으로 압축)이 각각 무엇을 잃는지 비교합니다.

In [12]:
# 가짜 긴 대화 — 예산 정보는 '앞부분'에 있다
long_history = [
    {"role":"user","content":"내 예산은 1인당 15000원이야."},
    {"role":"assistant","content":"예산 15000원 기억했습니다."},
] + sum([[{"role":"user","content":f"잡담 {i}: 오늘 날씨 얘기"},
          {"role":"assistant","content":f"네, 잡담 {i} 답변입니다."}] for i in range(1,6)], [])

def ask_with(history, q):
    r = client.chat.completions.create(model=MODEL,
        messages=history + [{"role":"user","content":q}])
    return r.choices[0].message.content

q = "아까 내 예산이 1인당 얼마라고 했지?"
trimmed = long_history[-4:]                              # Trimming: 최근 4턴만
summary = summarize(long_history)                        # Summarization: 압축
sum_ctx = [{"role":"system","content":f"지금까지 대화 요약: {summary}"}]

print("[Trimming]", ask_with(trimmed, q))
print("[Summarization]", ask_with(sum_ctx, q))
# 관찰: 잘라내기는 오래된-중요 정보(예산)를 잃고, 요약은 요지를 보존한다

[Trimming] 죄송합니다. 이전 대화의 내용은 기억할 수 없기 때문에 아까 언급된 예산에 대한 정보를 확인할 수 없습니다. 예산에 대해 다시 말씀해 주시면 도움이 될 수 있습니다!
[Summarization] 사용자의 예산은 1인당 15,000원이라고 하셨습니다.


## 6단계 — 메모리도 '검색'이 된다 (경험의 임베딩 검색)

강의의 "메모리 vs RAG — 기법은 같고 대상이 다르다". 이번엔 외부 문서가 아니라 **내 과거 발화**를 임베딩 검색합니다.

In [13]:
mem_texts = [m["content"] for m in long_history if m["role"] == "user"]
mem_vecs = embed(mem_texts)

def search_memory(q, k=1):
    qv = embed([q])[0]
    scored = sorted(((cosine(qv, v), t) for v, t in zip(mem_vecs, mem_texts)), reverse=True)
    return scored[:k]

for score, text in search_memory("돈을 얼마나 쓸 수 있다고 했더라?"):
    print(f"{score:.3f}  {text}")
# 관찰: '예산'이라는 단어 없이도 의미로 과거 발화를 찾는다 — 개인화의 토대

0.333  내 예산은 1인당 15000원이야.


## 7단계 — Agentic RAG 맛보기: 검색할지 말지 스스로 판단

지금까지는 매번 검색했습니다. 에이전트에게 **검색 필요 여부**를 먼저 판단시켜 봅니다 — RAG를 '도구'로 쓰는 첫걸음.

In [17]:
def answer_agentic(q):
    judge = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role":"system","content":"'저장된 맛집 메모'가 있으면 개인적으로 답변할 수 있는가? yes/no 한 단어만."},
        {"role":"user","content":q}]).choices[0].message.content.strip().lower()
    if judge.startswith("yes"):
        print(f"  (판단: 검색 필요 → RAG 실행)")
        return answer(q)
    print(f"  (판단: 검색 불필요 → 바로 답)")
    return client.chat.completions.create(model=MODEL,
        messages=[{"role":"user","content":q}]).choices[0].message.content

print(answer_agentic("혼자 가기 좋은 맛집 있어?"))          # 메모 검색 필요
print()
print(answer_agentic("고마워! 좋은 하루 보내"))             # 검색 불필요
# 관찰: 7차시 ReAct 루프에 '검색 도구'를 꽂은 것과 같은 구조

  (판단: 검색 필요 → RAG 실행)
📚 근거로 찾은 메모:
  (0.397) 순한국밥집: 강남. 담백한 소고기국밥, 해장에 좋음. 9천원. 24시간 영업.
  (0.397) 라멘하루: 홍대. 돈코츠 라멘이 진하고 혼밥하기 편함. 1.1만원. 웨이팅 있음.

🤖 추천: 홍대에 있는 '라멘하루'를 추천합니다. 돈코츠 라멘이 진하고 혼밥하기 편한 곳입니다. 가격은 1.1만원이며, 웨이팅이 있을 수 있습니다.
None

  (판단: 검색 불필요 → 바로 답)
천만에요! 당신도 좋은 하루 되세요! 도움이 필요하면 언제든지 말씀해 주세요.


## 8단계 — ⭐ 내 팀 MVP에 적용

위 구조(대화 메모리 → 임베딩 검색 → 근거 있는 답)는 **그대로** 당신 팀 MVP에 옮길 수 있습니다.
맛집 메모 자리에 **우리 서비스의 데이터/노트**를 넣기만 하면 됩니다. (예: 일정앱→일정 메모, 가계부→지출 내역)

⌨️ **터미널 Codex:** `codex "우리 MVP 데이터를 임베딩 검색해 근거 있는 답을 내도록 붙이고, 대화 메모리로 선호를 기억하게 해줘"`

In [ ]:
# 팀별 작성: 우리 MVP가 가진 '데이터/노트'를 적어보세요 (검색 대상)
MY_NOTES_TODO = [
    # "항목1: ...설명...",
    # "항목2: ...설명...",
]

if MY_NOTES_TODO:
    notes = MY_NOTES_TODO                 # 우리 데이터로 교체
    note_vecs = embed(notes)              # 다시 임베딩(인제스션)
    print("우리 데이터로 교체 완료:", len(notes), "건 → answer('우리 도메인 질문')으로 확인")
else:
    print("⬜ MY_NOTES_TODO를 채운 뒤 다시 실행하면 '내 MVP 검색 에이전트' 완성")
    print("→ notes/note_vecs 교체 후 answer(\"우리 서비스 관련 질문\")")

## 정리

- **메모리** = 에이전트가 **겪은 일**을 기억 (단기: 대화 누적·롤링 윈도우 / 장기: 요약·영속).
- **지식 = RAG** = **검색(R) + 생성(G)** — 모델 재학습 없이 **내 데이터**를 답에 녹이고, **출처**까지 제시.
- **임베딩**으로 텍스트를 의미 벡터로 바꾸고, **코사인 유사도**로 의미가 가까운 메모를 찾는다(의미 검색).
- **메모리 vs RAG**: 기법은 같아도 검색 **대상**이 다름 — 내 과거 대화(메모리) vs 외부/우리 데이터(RAG).
- 핵심은 **컨텍스트에 알맞은 재료(기억 + 검색된 지식)를 채워 넣기** — 출력이 아니라 **근거 있는 결과**.